# F1 Strategy Recommender (Local OpenF1 CSV + Separate Forecast CSV)

This notebook:
1. Builds a race-level training dataset from local `openf1_data/`
2. Trains two models:
   - pit stop count classifier
   - target tire compound classifier
3. Uses separate forecast weather CSV for inference

**Expected folder structure:**
- `openf1_data/` (your local files)
- this notebook

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow import keras
from tensorflow.keras import layers

## 1) Config

In [2]:
DATA_DIR = Path("openf1_data")
TRAIN_DATASET_FILE = "race_strategy_dataset.csv"
STOPS_MODEL_FILE = "stops_model.keras"
TIRE_MODEL_FILE = "tire_model.keras"
PREPROC_FILE = "preprocessing.joblib"

RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather.csv"

assert DATA_DIR.exists(), f"Missing directory: {DATA_DIR.resolve()}"

## 2) Helpers for local OpenF1 files

In [3]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None


def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()


def safe_mode(series, default="MEDIUM"):
    s = pd.Series(series).dropna()
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else default


def load_sessions():
    p = DATA_DIR / "sessions.csv"
    if p.exists():
        return pd.read_csv(p)
    return pd.DataFrame()


def load_meetings_map():
    meeting_files = list(DATA_DIR.glob("meetings_meeting_*.csv"))
    rows = []
    for f in meeting_files:
        df = safe_read_csv(f)
        if df.empty:
            continue
        for c in ["meeting_key", "circuit_short_name", "location", "country_name", "meeting_name"]:
            if c not in df.columns:
                df[c] = np.nan
        rows.append(df[["meeting_key", "circuit_short_name", "location", "country_name", "meeting_name"]].head(1))

    if not rows:
        return pd.DataFrame(columns=["meeting_key", "track_name"])

    mdf = pd.concat(rows, ignore_index=True).drop_duplicates(subset=["meeting_key"])
    mdf["track_name"] = (
        mdf["circuit_short_name"]
        .fillna(mdf["location"])
        .fillna(mdf["meeting_name"])
        .fillna(mdf["country_name"])
        .fillna(mdf["meeting_key"].astype(str))
    )
    return mdf[["meeting_key", "track_name"]]


def weather_features_for_session(session_key: int):
    wf = DATA_DIR / f"weather_session_{session_key}.csv"
    w = safe_read_csv(wf)
    if w.empty:
        return None

    needed = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
    for c in needed:
        if c not in w.columns:
            w[c] = np.nan
        w[c] = pd.to_numeric(w[c], errors="coerce")

    w = w.dropna(subset=needed, how="all")
    if w.empty:
        return None

    return {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "air_temp_max": float(w["air_temperature"].max()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "track_temp_max": float(w["track_temperature"].max()),
        "humidity_mean": float(w["humidity"].mean()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
        "rainfall_sum": float(w["rainfall"].sum()),
        "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean())
    }

## 3) Build race-level dataset from local files

In [4]:
def get_session_track_map_and_meeting_map():
    sessions = load_sessions()
    meetings = load_meetings_map()

    if sessions.empty:
        return {}, {}

    for c in ["session_key", "meeting_key", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    s = sessions.copy().merge(meetings, on="meeting_key", how="left")
    s["track_name_final"] = (
        s["circuit_short_name"]
        .fillna(s["location"])
        .fillna(s["track_name"])
        .fillna(s["meeting_key"].astype(str))
    )

    session_to_track = dict(zip(s["session_key"], s["track_name_final"]))
    session_to_meeting = dict(zip(s["session_key"], s["meeting_key"]))
    return session_to_track, session_to_meeting


def build_local_dataset():
    stints_files = sorted(DATA_DIR.glob("stints_session_*.csv"))
    if not stints_files:
        raise FileNotFoundError("No stints_session_*.csv files found.")

    session_to_track, session_to_meeting = get_session_track_map_and_meeting_map()
    rows = []

    for sf in stints_files:
        sid = parse_session_key(sf)
        if sid is None:
            continue

        stints = safe_read_csv(sf)
        if stints.empty:
            continue

        for c in ["driver_number", "compound", "stint_number"]:
            if c not in stints.columns:
                stints[c] = np.nan

        drivers = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if drivers.empty:
            continue
        for c in ["driver_number", "team_name"]:
            if c not in drivers.columns:
                drivers[c] = np.nan

        driver_team = (
            drivers[["driver_number", "team_name"]]
            .dropna(subset=["driver_number", "team_name"])
            .drop_duplicates()
        )
        stints = stints.merge(driver_team, on="driver_number", how="left")

        pits = safe_read_csv(DATA_DIR / f"pit_session_{sid}.csv")
        if pits.empty:
            pits = pd.DataFrame(columns=["driver_number"])
        if "driver_number" not in pits.columns:
            pits["driver_number"] = np.nan
        pits = pits.merge(driver_team, on="driver_number", how="left")

        grid = safe_read_csv(DATA_DIR / f"starting_grid_session_{sid}.csv")
        grid_driver_col = "driver_number" if "driver_number" in grid.columns else None
        grid_pos_col = None
        for cand in ["position", "grid_position"]:
            if cand in grid.columns:
                grid_pos_col = cand
                break

        if grid_driver_col and grid_pos_col:
            g = grid[[grid_driver_col, grid_pos_col]].copy()
            g.columns = ["driver_number", "grid_pos"]
            g["grid_pos"] = pd.to_numeric(g["grid_pos"], errors="coerce")
            g = g.merge(driver_team, on="driver_number", how="left")
            team_starting_pos = g.groupby("team_name", dropna=True)["grid_pos"].mean().to_dict()
        else:
            team_starting_pos = {}

        wf = weather_features_for_session(sid)
        if wf is None:
            continue

        track_name = session_to_track.get(sid, str(session_to_meeting.get(sid, "UNKNOWN_TRACK")))

        for team, t_stints in stints.groupby("team_name", dropna=True):
            if pd.isna(team):
                continue

            t_pits = pits[pits["team_name"] == team]

            stint1 = t_stints[t_stints["stint_number"] == 1]
            if len(stint1):
                starting_compound = safe_mode(stint1["compound"], default=safe_mode(t_stints["compound"]))
            else:
                starting_compound = safe_mode(t_stints["compound"], default="MEDIUM")

            non_start = t_stints[t_stints["compound"] != starting_compound]["compound"]
            target_compound = safe_mode(non_start, default=safe_mode(t_stints["compound"], default="MEDIUM"))

            starting_position = team_starting_pos.get(team, np.nan)
            if pd.isna(starting_position):
                starting_position = 10.0

            rows.append({
                "session_key": sid,
                "meeting_key": session_to_meeting.get(sid, np.nan),
                "team_name": str(team),
                "track_name": str(track_name),
                "starting_position": float(starting_position),
                "starting_compound": str(starting_compound),
                "pit_stops": int(len(t_pits)),
                "target_compound": str(target_compound),
                **wf
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Dataset is empty. Check files/columns.")

    df["starting_position"] = pd.to_numeric(df["starting_position"], errors="coerce").fillna(10.0)
    for c in ["team_name", "track_name", "starting_compound", "target_compound"]:
        df[c] = df[c].fillna("UNKNOWN").astype(str)

    df.to_csv(TRAIN_DATASET_FILE, index=False)
    return df


df_train = build_local_dataset()
print(f"Saved {TRAIN_DATASET_FILE} with {len(df_train)} rows")
df_train.head()

Saved race_strategy_dataset.csv with 580 rows


,session_key,meeting_key,team_name,track_name,starting_position,starting_compound,pit_stops,target_compound,air_temp_mean,air_temp_max,track_temp_mean,track_temp_max,humidity_mean,wind_speed_mean,rainfall_sum,rain_minutes_ratio
0,10029,1259,Alpine,Miami,17.0,SOFT,8,SOFT,27.974359,28.8,41.015385,46.9,63.051282,2.519231,0.0,0.0
1,10029,1259,Aston Martin,Miami,17.5,SOFT,6,SOFT,27.974359,28.8,41.015385,46.9,63.051282,2.519231,0.0,0.0
2,10029,1259,Ferrari,Miami,10.0,SOFT,11,SOFT,27.974359,28.8,41.015385,46.9,63.051282,2.519231,0.0,0.0
3,10029,1259,Haas F1 Team,Miami,14.0,SOFT,9,SOFT,27.974359,28.8,41.015385,46.9,63.051282,2.519231,0.0,0.0
4,10029,1259,Kick Sauber,Miami,14.5,SOFT,8,SOFT,27.974359,28.8,41.015385,46.9,63.051282,2.519231,0.0,0.0


## 4) Train models

In [5]:
NUMERIC_COLS = [
    "starting_position",
    "air_temp_mean",
    "air_temp_max",
    "track_temp_mean",
    "track_temp_max",
    "humidity_mean",
    "wind_speed_mean",
    "rainfall_sum",
    "rain_minutes_ratio",
]
CAT_COLS = ["team_name", "track_name", "starting_compound"]


def build_model(cat_cards, n_num, n_classes):
    cat_inputs, cat_vecs = [], []
    for i, card in enumerate(cat_cards):
        inp = keras.Input(shape=(1,), dtype="int32", name=f"cat_{i}")
        dim = min(16, max(4, card // 2))
        emb = layers.Embedding(card, dim)(inp)
        emb = layers.Reshape((dim,))(emb)
        cat_inputs.append(inp)
        cat_vecs.append(emb)

    num_inp = keras.Input(shape=(n_num,), dtype="float32", name="num_input")
    x = layers.Concatenate()(cat_vecs + [num_inp])
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    m = keras.Model(cat_inputs + [num_inp], out)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


df = pd.read_csv(TRAIN_DATASET_FILE).copy()

for c in CAT_COLS:
    df[c] = df[c].fillna("UNKNOWN").astype(str)
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

encoders = {}
for c in CAT_COLS:
    le = LabelEncoder()
    df[c + "_enc"] = le.fit_transform(df[c])
    encoders[c] = le

stops_encoder = LabelEncoder()
tire_encoder = LabelEncoder()

y_stops = stops_encoder.fit_transform(df["pit_stops"])
y_tire = tire_encoder.fit_transform(df["target_compound"])

X_cat = [df[c + "_enc"].values for c in CAT_COLS]
X_num = df[NUMERIC_COLS].values.astype("float32")

(
    c0_tr, c0_te,
    c1_tr, c1_te,
    c2_tr, c2_te,
    n_tr, n_te,
    ys_tr, ys_te,
    yt_tr, yt_te
) = train_test_split(
    X_cat[0], X_cat[1], X_cat[2], X_num, y_stops, y_tire,
    test_size=0.2, random_state=42
)

train_in = [c0_tr, c1_tr, c2_tr, n_tr]
test_in = [c0_te, c1_te, c2_te, n_te]
cat_cards = [len(encoders[c].classes_) for c in CAT_COLS]

stops_model = build_model(cat_cards, len(NUMERIC_COLS), len(stops_encoder.classes_))
stops_model.fit(train_in, ys_tr, validation_split=0.2, epochs=30, batch_size=32, verbose=1)
print("Stops test:", stops_model.evaluate(test_in, ys_te, verbose=0))
stops_model.save(STOPS_MODEL_FILE)

tire_model = build_model(cat_cards, len(NUMERIC_COLS), len(tire_encoder.classes_))
tire_model.fit(train_in, yt_tr, validation_split=0.2, epochs=30, batch_size=32, verbose=1)
print("Tire test:", tire_model.evaluate(test_in, yt_te, verbose=0))
tire_model.save(TIRE_MODEL_FILE)

joblib.dump(
    {
        "encoders": encoders,
        "stops_encoder": stops_encoder,
        "tire_encoder": tire_encoder,
        "numeric_cols": NUMERIC_COLS,
        "cat_cols": CAT_COLS,
    },
    PREPROC_FILE,
)
print("Saved models + preprocessing artifacts")

2026-04-27 12:05:42.803643: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-04-27 12:05:42.803680: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-04-27 12:05:42.803686: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-04-27 12:05:42.803701: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-27 12:05:42.803711: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/30


2026-04-27 12:05:43.326563: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.1159 - loss: 22.4436 - val_accuracy: 0.2903 - val_loss: 12.9750
Epoch 2/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1482 - loss: 14.6108 - val_accuracy: 0.2796 - val_loss: 7.2327
Epoch 3/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1563 - loss: 13.8737 - val_accuracy: 0.3011 - val_loss: 9.0440
Epoch 4/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1617 - loss: 12.8910 - val_accuracy: 0.2366 - val_loss: 7.3327
Epoch 5/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1725 - loss: 12.0851 - val_accuracy: 0.3011 - val_loss: 6.8201
Epoch 6/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1563 - loss: 11.6372 - val_accuracy: 0.2581 - val_loss: 5.4986
Epoch 7/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1482 - loss: 11.8172 - val_accuracy: 0.2043 - val_loss: 5.5741
Epoch 8/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1833 - loss: 11.3206 - val_accuracy: 0.3011 - val

## 5) Create example inference files (separate race context + forecast weather)

In [6]:
race_context_example = pd.read_csv(RACE_CONTEXT_FILE)
#race_context_example.to_csv(RACE_CONTEXT_FILE, index=False)

forecast_example = pd.read_csv(FORECAST_FILE)

#forecast_example.to_csv(FORECAST_FILE, index=False)

print(f"Wrote {RACE_CONTEXT_FILE} and {FORECAST_FILE}")

Wrote race_context_input.csv and forecast_weather.csv


## 6) Inference with separate forecast CSV

In [7]:
def encode_with_unknown(le, value):
    value = str(value)
    if value in set(le.classes_):
        return int(le.transform([value])[0])
    return 0


def forecast_to_features(forecast_df):
    required = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
    miss = [c for c in required if c not in forecast_df.columns]
    if miss:
        raise ValueError(f"{FORECAST_FILE} missing columns: {miss}")

    f = forecast_df.copy()
    for c in required:
        f[c] = pd.to_numeric(f[c], errors="coerce")

    f = f.dropna(subset=required)
    if f.empty:
        raise ValueError("forecast file has no valid weather rows")

    return {
        "air_temp_mean": float(f["air_temperature"].mean()),
        "air_temp_max": float(f["air_temperature"].max()),
        "track_temp_mean": float(f["track_temperature"].mean()),
        "track_temp_max": float(f["track_temperature"].max()),
        "humidity_mean": float(f["humidity"].mean()),
        "wind_speed_mean": float(f["wind_speed"].mean()),
        "rainfall_sum": float(f["rainfall"].sum()),
        "rain_minutes_ratio": float((f["rainfall"] > 0).mean()),
    }


art = joblib.load(PREPROC_FILE)
encoders = art["encoders"]
stops_encoder = art["stops_encoder"]
tire_encoder = art["tire_encoder"]
numeric_cols = art["numeric_cols"]

stops_model = keras.models.load_model(STOPS_MODEL_FILE)
tire_model = keras.models.load_model(TIRE_MODEL_FILE)

rc = pd.read_csv(RACE_CONTEXT_FILE).iloc[0]
fc = pd.read_csv(FORECAST_FILE)
wf = forecast_to_features(fc)

team_enc = encode_with_unknown(encoders["team_name"], rc["team_name"])
track_enc = encode_with_unknown(encoders["track_name"], rc["track_name"])
comp_enc = encode_with_unknown(encoders["starting_compound"], rc["starting_compound"])

feature_row = {"starting_position": float(rc["starting_position"]), **wf}
num_vals = np.array([feature_row[c] for c in numeric_cols], dtype=np.float32).reshape(1, -1)

model_in = [
    np.array([team_enc], dtype=np.int32),
    np.array([track_enc], dtype=np.int32),
    np.array([comp_enc], dtype=np.int32),
    num_vals,
]

sp = stops_model.predict(model_in, verbose=0)[0]
tp = tire_model.predict(model_in, verbose=0)[0]

s_cls = int(np.argmax(sp))
t_cls = int(np.argmax(tp))

print("\n--- STRATEGY RESULT (FORECAST-BASED) ---")
print("Team:", rc["team_name"])
print("Track:", rc["track_name"])
print("Starting position:", rc["starting_position"])
print("Starting compound:", rc["starting_compound"])
print("\nForecast summary:")
for k, v in wf.items():
    print(f"  {k}: {v:.3f}")

print("\nRecommendations:")
print("  Pit stops:", stops_encoder.inverse_transform([s_cls])[0], f"(conf={sp[s_cls]:.3f})")
print("  Next/main compound:", tire_encoder.inverse_transform([t_cls])[0], f"(conf={tp[t_cls]:.3f})")


--- STRATEGY RESULT (FORECAST-BASED) ---
Team: Ferrari
Track: Imola
Starting position: 4
Starting compound: MEDIUM

Forecast summary:
  air_temp_mean: 22.767
  air_temp_max: 23.200
  track_temp_mean: 38.950
  track_temp_max: 40.000
  humidity_mean: 41.167
  wind_speed_mean: 2.633
  rainfall_sum: 0.100
  rain_minutes_ratio: 0.167

Recommendations:
  Pit stops: 8 (conf=0.600)
  Next/main compound: SOFT (conf=0.989)
